# Ablaciones y paridad de exportación nativa: fundamentos

**Nivel:** 1  
**Requisitos previos:** 05-evaluation-and-transfer  
**Tiempo de lectura:** 15 minutos

## Objetivos

El objetivo es explicar el problema y sus datos, inspeccionar y, en niveles superiores, ejecutar el método publicado en su tarea emparejada. Una evaluación completa debe comprobar si cada componente propuesto contribuye y si las exportaciones optimizadas del modelo preservan el puntuador de referencia. La coincidencia de puntajes es evidencia numérica; no demuestra decisiones más seguras, mejores resultados de tarea ni generalización.

## Preparación

Las versiones siguientes están fijadas en `notebooks/requirements.txt` y también constan en los metadatos de este notebook.

```bash
pip install reflexmesh==0.1.2 numpy==2.4.6 matplotlib==3.11.2 torch==2.14.1 joblib==1.6.0 gymnasium==1.4.0 scikit-learn==1.9.1 stable-baselines3==2.9.2 sb3-contrib==2.9.2
```

## Datos

El notebook lee artefactos comprometidos del producto. Propietario: Neuraxis. Repositorio: https://github.com/fsantibanezleal/CAOS_Neuraxis. Licencia: Apache-2.0. El JSON es un artefacto experimental propio y congelado; este notebook no redistribuye datos de benchmarks externos.

Este notebook lee `data/derived/index.json` y las trazas listadas. La ejecución usa un espacio aislado de caso Reflexmesh. Para métodos aprendidos, descarga los activos públicos exactos con `python -m reflexmesh.cli fetch-models --destination models/checkpoints --version v0.01.000` y define `NEURAXIS_CHECKPOINTS` hacia esa carpeta. El manifiesto de publicación verifica los hashes. M09/M10 se leen desde trazas congeladas, salvo que `NEURAXIS_NOTEBOOK_MODEL` indique explícitamente un modelo Ollama local configurado.

Artefactos de esta lección:

- `data/derived/index.json`
- `data/derived/research.json`
- `docs/evaluation/corrected-results.json`
- `data/derived/episodes/M03-C07-nominal-40000.json`
- `data/derived/episodes/M04-C07-nominal-40000.json`
- `data/derived/episodes/M11-C07-nominal-40000.json`
- `data/derived/episodes/M12-C07-nominal-40000.json`

## Método

Audita el digest del índice canónico frente al informe de investigación, compara las cuatro ablaciones emparejadas de M12 con el controlador completo y luego inspecciona los vectores de exportación nativa M03/M04, cobertura de umbrales, tolerancia de error y alcance de latencia en caliente.

Para el resultado emparejado \(Y\), el contraste de ablación es \(\Delta=Y_{completo}-Y_{ablacionado}\); las semillas emparejadas y las plantillas fijas determinan la unidad de incertidumbre. La paridad de exportación es \(\max_i |p_i^{nativo}-p_i^{referencia}|\le\epsilon\), un criterio numérico separado.

### Comprueba las versiones fijadas del entorno.

In [1]:
# Setup is performed outside the notebook; report the exact runtime pins.
import importlib.metadata
for distribution in ('reflexmesh', 'numpy', 'matplotlib', 'torch', 'joblib', 'gymnasium', 'scikit-learn', 'stable-baselines3', 'sb3-contrib'):
    print('{}=={}'.format(distribution, importlib.metadata.version(distribution)))

reflexmesh==0.1.2
numpy==2.4.6
matplotlib==3.11.2
torch==2.14.1
joblib==1.6.0
gymnasium==1.4.0
scikit-learn==1.9.1
stable-baselines3==2.9.2
sb3-contrib==2.9.2


### Verifica la auditoría y los bytes de cada traza antes de interpretar resultados.

In [2]:
# Lee la evidencia comprometida y usa la API publicada; nunca define otra política.
import hashlib, json
from pathlib import Path
ROOT = Path('../../../').resolve()
index_path = ROOT / 'data/derived/index.json'
index_bytes = index_path.read_bytes()
index = json.loads(index_bytes)
audit = json.loads((ROOT / 'data/derived/benchmark-adequacy.json').read_text(encoding='utf-8'))
assert hashlib.sha256(index_bytes).hexdigest() == audit['index_sha256']
selected_methods = ['M03', 'M04', 'M11', 'M12']
sample_case = 'C07'
seed = 40000
records = {}
for method_id in selected_methods:
    row = next(r for r in index['episodes'] if r['method_id'] == method_id and r['case_id'] == sample_case and r['variant'] == 'nominal' and r['seed'] == seed)
    trace_path = ROOT / 'data/derived' / row['path']
    trace_bytes = trace_path.read_bytes()
    assert hashlib.sha256(trace_bytes).hexdigest() == row['sha256']
    records[method_id] = json.loads(trace_bytes)
print('Index digest:', hashlib.sha256(index_bytes).hexdigest())
print('Evidence records:', len(records), '| case:', sample_case, '| seed:', seed)
print('Methods:', ', '.join(selected_methods))

Index digest: 1024a451fdc1f1aebc4e8d8d66e6bf1aca119f26ae058d3de9b84ff312d8d7a6
Evidence records: 4 | case: C07 | seed: 40000
Methods: M03, M04, M11, M12


### Inspecciona las decisiones y resultados originales

In [3]:
# Resume la primera acción registrada, los candidatos autorizados y el resultado del oráculo independiente.
for method_id, episode in records.items():
    first = episode['decisions'][0]
    allowed = [candidate['action'] for candidate in first['candidates'] if candidate['allowed']]
    print(method_id, '| action=', first['action'], '| reason=', first['reason'], '| allowed=', allowed)
    print('  success=', episode['metrics']['task_success'], '| steps=', episode['metrics']['steps'], '| violations=', episode['metrics']['violations'])

M03 | action= a-cancel | reason= calibrated_demonstration_preference | allowed= ['a-ack_event', 'a-cancel', 'a-refresh', 'a-report', 'a-wait', 'a-wait_dependency']
  success= 1 | steps= 1 | violations= 0
M04 | action= a-cancel | reason= calibrated_demonstration_preference | allowed= ['a-ack_event', 'a-cancel', 'a-refresh', 'a-report', 'a-wait', 'a-wait_dependency']
  success= 1 | steps= 1 | violations= 0
M11 | action= a-cancel | reason= learned_incremental_benefit_gate | allowed= ['a-ack_event', 'a-cancel', 'a-refresh', 'a-report', 'a-wait', 'a-wait_dependency']
  success= 1 | steps= 1 | violations= 0
M12 | action= a-cancel | reason= urgent_cancellation_during_deliberation | allowed= ['a-ack_event', 'a-cancel', 'a-refresh', 'a-report', 'a-wait', 'a-wait_dependency']
  success= 1 | steps= 1 | violations= 0


## Resultados

Las cuatro ablaciones conservan los mismos 1.160/1.200 éxitos de tarea, así que este experimento no muestra contribución al éxito por memoria, efectos, aplazamiento o invalidación. Los presupuestos de ejecución difieren entre variantes, y las latencias medidas excluyen la construcción de características en Python. La paridad numérica no calibra la seguridad de la política.

Este paso de autoría no compromete salidas generadas; `scripts/run_notebooks.py` las ejecuta y guarda desde un kernel nuevo.

## Resumen

El resultado negativo de ablación importa: los componentes propuestos no cambian el éxito en esta suite saturada. Las exportaciones nativas cumplen la tolerancia numérica registrada, lo que respalda solo consistencia de implementación.

## Referencias

- [XGB](https://arxiv.org/abs/1603.02754)
- [PPO](https://arxiv.org/abs/1707.06347)